 This notebook explores LangChain tool calling through the implementation of an AI math assistant, documenting my practical learning in agent development, custom tool integration, and natural language interaction.

## Installing required libraries



In [ ]:
%pip install langchain>=1.0 | tail -n 1 
%pip install langchain-ibm==1.1.1 | tail -n 1 
%pip install langchain-community==0.4.2 | tail -n 1 
%pip install wikipedia==1.4.0 | tail -n 1
%pip install openai==1.77.0 | tail -n 1
%pip install langchain-openai==1.6.6 | tail -n 1

## Import the required libraries


In [ ]:
from langchain_ibm import ChatWatsonx
from ibm_watsonx_ai.wml_resource import WatsonxAPIWarning
from langchain.agents import create_agent

import re
import warnings
warnings.filterwarnings("ignore", category=WatsonxAPIWarning)

## Loading the LLM: Choosing the right language model

This project uses IBM’s ChatWatsonx to connect a language model with tools, with configurable parameters such as temperature and max_tokens to control response randomness and length.


In [ ]:
llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
)

Let's generate a simple response:


In [ ]:
response = llm.invoke("What is tool calling in langchain?")
print("\nResponse Content: ", response.content)

## API Disclaimer

For local execution, configure your own API credentials and update all ChatOpenAI and ChatWatsonx instances using the configurations below. Run this setup cell only when executing the notebook locally.


## Function


Tools expose functions that an AI agent can call to perform specific tasks, using structured inputs, consistent outputs, and clear documentation. The addition function below demonstrates these principles but requires input validation to handle non-numeric values reliably.


In [ ]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

Directly testing a tool allows to pinpoint where the problem lies—whether it’s in the tool’s logic, input parsing, or output formatting.
Here,  input the string `1 2` and get the sum.


In [ ]:
add_numbers("1 2")


## Tool
LangChain’s Tool class wraps a Python function for use by an AI agent, combining a name, the underlying function, and a description that guides the agent on when to use it.




In [ ]:
from langchain_core.tools import Tool
add_tool=Tool(
        name="AddTool",
        func=add_numbers,
        description="Adds a list of numbers and returns the result.")

print("tool object",add_tool)


These attributes allow to inspect the tool object


In [ ]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)


call the tool's function via the ```add_tool``` object:


In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input))  # Example

### `@tool` operator

The @tool decorator simplifies tool creation by converting a Python function into an agent-compatible tool, using its name, type hints, and docstring to define the tool’s interface.


In [ ]:
from langchain_core.tools import tool
import re

@tool
def add_numbers(inputs: str) -> dict:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return {"result": sum(numbers)}

The above function will now act as a tool. 


In [ ]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 


calling the tool using the ```invoke``` method.


In [ ]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))  # Example


### Use @tool-StructuredTool

The @tool decorator creates a StructuredTool, inferring its input schema from the function signature and using the docstring as its description. This helps language models identify the required arguments and call the tool correctly.


In [ ]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

In [ ]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

Both tools accept an inputs string for numeric extraction and summation. add_numbers_with_options also includes an optional absolute boolean, defaulting to False, which enables taking each number’s absolute value before summing.

In [ ]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

Pass a dictionary with keys matching the tool’s parameter names. In this example, setting absolute=False returns -6, while absolute=True sums the absolute values and returns 6.

In [ ]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

## Improved tool return types with Python typing

When creating tools, we have to accurately specify their return values. This helps the agent understand and handle different possible outputs.



sum_numbers_with_complex_output returns a dictionary containing either the calculated sum as a float or a descriptive error message if no numbers are found or processing fails.

In [ ]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}


sum_numbers_from_text extracts integers from an input string and returns their sum as a float, without explicit handling for missing numbers or processing errors.

In [ ]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return sum(numbers)

### `create_agent`

create_agent connects a language model with tools to interpret requests, select appropriate operations, and generate responses from tool results. Its key parameters include model, tools, an optional system_prompt to guide behavior, and debug for execution details.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add_tool]
)

Now, we can run the agent by asking a question.


In [ ]:
# Use the agent
response = agent.invoke({
    "messages": [("human", "In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")]
})
print(response["messages"][-1].content)

In [ ]:
response

In [ ]:
agent.invoke({"messages": [("human", "Add 10, 20, two and 30")]})

#### **Structured tool calling with `create_agent`**

create_agent supports both simple and multi-input tools defined with @tool, using structured input schemas to guide tool calls and incorporating returned values into the agent’s response.


The example below uses create_agent with the Granite language model and the sum_numbers_from_text tool to process natural language requests and calculate sums.

In [ ]:
agent_2 = create_agent(
    model=llm,
    tools=[sum_numbers_from_text]
)
response = agent_2.invoke({
    "messages": [("human", "Add -10, -20, and -30 using absolute values.")]
})
print(response["messages"][-1].content)

In [ ]:
from langchain_openai import ChatOpenAI

llm_ai = ChatOpenAI(model="gpt-5.4-nano")

In [ ]:
agent_3 = create_agent(
    model=llm_ai,
    tools=[sum_numbers_with_complex_output]
)
response = agent_3.invoke({"messages": [("human", "Add 10, 20 and 30")]})
print(response["messages"][-1].content)

This example uses create_agent with Granite and the multi-input add_numbers_with_options tool, allowing the agent to specify both the input text and the optional absolute-value setting.

In [ ]:
agent_2 = create_agent(
    model=llm,
    tools=[add_numbers_with_options]
)

In [ ]:
response = agent_2.invoke({
    "messages": [("Add -10, -20, and -30 using absolute values.")]
})
print(response)

Let's try with OpenAI to see if it runs with multiple inputs.


In [ ]:
agent_openai = create_agent(
    model=llm_ai,
    tools=[add_numbers_with_options]
)

In [ ]:
response = agent_openai.invoke({
    "messages": [("human", "Add -10, -20, and -30 using absolute values.")]
})
print(response)

### **`create_agent`**

create_agent builds an agent graph using a language model, available tools, and an optional system_prompt. It alternates between model calls and tool execution, adding tool results to the message history until the model produces a final response.


In [ ]:
%pip install langgraph==0.6.1 | tail -n 1

In [ ]:
from langchain.agents import create_agent

agent_exec = create_agent(model=llm_ai, tools=[sum_numbers_from_text])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})

In [ ]:
print(msgs["messages"][-1].content)

## Orchestrating multiple tools with an agent: Mathematical toolkit
The mathematical toolkit combines addition, subtraction, multiplication, and division tools, allowing the agent to select the appropriate operation for each request. The subtraction tool subtracts each subsequent number from the first, such as 100 − 20 − 10 = 70.


In [ ]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and successively subtracts each number after the first
    from the first number.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": 70}
    """
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 0}

    result = numbers[0]
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [ ]:
print("Name: \n", subtract_numbers.name)
print("Description: \n", subtract_numbers.description) 
print("Args: \n", subtract_numbers.args) 

Let's use it directly by calling the function:


In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(subtract_numbers.invoke(test_input))  # Example

The multiply_numbers and divide_numbers functions calculate the product and sequential division of numbers provided as text. The @tool decorator converts them into agent-compatible tools, using their docstrings as descriptions to guide tool selection.

In [ ]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """Extracts numbers from a string and calculates their product."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 1}

    result = 1
    for num in numbers:
        result *= num
    return {"result": result}

In [ ]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """Extracts numbers from a string and divides the first by each subsequent number."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 0}

    if any(num == 0 for num in numbers[1:]):
        return {"result": "Cannot divide by zero."}

    result = numbers[0]
    for num in numbers[1:]:
        result /= num
    return {"result": result}

In [ ]:
# Testing multiply_tool
multiply_test_input = "2, 3, and 4"
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

In [ ]:
# Testing divide_tool
divide_test_input = "100, 5, 2"
divide_result = divide_numbers.invoke(divide_test_input)
print("--- Testing DivideTool ---")
print(f"Input: {divide_test_input}")
print(f"Output: {divide_result}")

## Building the agent

Combine the addition, subtraction, multiplication, and division tools into a single list, then create an agent that selects the appropriate operation based on the user’s request.


In [ ]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

In [ ]:
# Create the agent with all tools
math_agent = create_agent(
    model=llm_ai,
    tools=tools,
    system_prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)

In [ ]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

In [ ]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)


SubtractTool unexpectedly negates the first number, returning -130 for 100, 20, 10 instead of 70. This mismatch causes repeated unsuccessful attempts until the agent reaches its execution limit; the remaining tools are tested before correcting the subtraction logic.

In [ ]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [("human", "Multiply 2, 3, and 4.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [("human", "Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)

Now lets change the `SubtractTool` so it subtracts the numbers directly (without negating the first number). This aligns the tool’s behavior with standard arithmetic and the agent’s expectations.


In [ ]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}


## Note: Tool naming when demonstrating different approaches

add_tool and add_numbers demonstrate two ways to create an addition tool: the Tool() constructor and the @tool decorator. In production, use clearly differentiated tools and avoid redundant functionality that can make tool selection ambiguous.


In [ ]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent_new = create_agent(
    model=llm,
    tools=tools_updated,
    system_prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent", math_agent_new)

Define test cases in a Python dictionary to evaluate the agent across standard calculations, division by zero, large numbers, decimals, and mixed operations. Automated testing helps identify errors and verify consistent behavior across different inputs.

In [ ]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

The code extracts the computation result from the agent’s message history by locating the relevant ToolMessage and parsing its JSON content, allowing the result to be compared with the expected value.

In [ ]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)

Improve all arithmetic tools with decimal support, strict input validation, and clear error messages for empty inputs, non-numeric values, and division by zero. Expand tests to cover mixed text and numbers, decimals, and sequential operations to verify reliable behavior.

## **Exploring LangChain's built-in tools**

While creating custom tools is powerful, LangChain provides a rich ecosystem of **pre-built tools** that solve common tasks out of the box. These tools abstract away complex implementation details (API calls, input parsing, error handling) and help to focus on building robust agents quickly.



#### **Using the Wikipedia tool**



In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia.wikipedia as wp
import requests as _requests
import time

wp.API_URL = "https://en.wikipedia.org/w/api.php"

def _patched_request(params, _retries=3):
    params['format'] = 'json'
    if 'action' not in params:
        params['action'] = 'query'
    headers = {"User-Agent": "AI-Math-Assistant-Lab/1.0 (contact@example.com)"}

    last_exc = None
    for attempt in range(_retries):
        r = _requests.get(wp.API_URL, params=params, headers=headers)
        try:
            return r.json()
        except ValueError as e:
            last_exc = e
            print(f"Attempt {attempt+1}/{_retries} failed — status {r.status_code}, body: {r.text[:200]!r}")
            time.sleep(1)
    raise last_exc

wp._wiki_request = _patched_request

# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [ ]:
search_wikipedia.invoke("What is tool calling?")

In [ ]:
# Update tools list to include the Wikipedia tool
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_agent(
    model=llm_ai,
    tools=tools_updated,
    system_prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

Test the agent with a multi-step query that retrieves real-world data using search_wikipedia and processes the retrieved values with multiply_numbers.

In [ ]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")

## **Creating a power tool to calculate exponents**

Define calculate_power to extract a base and exponent from an input string, compute \(x^y\), and return the value in a dictionary under the "result" key. Integrate the tool into the agent and test its functionality.


In [ ]:
def calculate_power(input_text: str) ->dict:
    """
    calculate the power of a number x^y)
    Parameters:
    - input_text(str): like "2", "3"
    Returns:
    - dict: {result} or error message
    """

    #try to get expression like 5^2
    match = re.search(r"(\d+(?:\.\d+)?)\s*\^+\s*(\d+(?:\.\d+)?)", input_text)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}
    # Try to extract expressions like "2 to the power of 3"
    match = re.search(r"(\d+(?:\.\d+)?)\s*\^+\s*(\d+(?:\.\d+)?)", input_text)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}
    # Fallback: assume two numbers separated by space or comma
    try:
        numbers = [float(num) for num in input_text.replace(",", " ").split()]
        if len(numbers) != 2:
            return {"result": "Invalid input. Please provide exactly two numbers."}
        base, exponent = numbers
        return {"result": base ** exponent}
    except ValueError:
        return {"result": "Invalid input format. Provide input like '2 3', '2^3', or '2 to the power of 3'."}



Applying the @tool decorator to calculate_power, using the function name and docstring to identify the tool and describe its purpose and expected input.

In [ ]:

from langchain_core.tools import tool

@tool
def power_tool(input_text: str) -> dict:
    """Calcuates the power of a number (x^y) """
    return calculate_power(input_text)


Creating an agent using create_agent, passing the previously configured language model and including power_tool in its list of available tools.

In [ ]:

tools = [power_tool]

agen = create_agent(
    model= llm,
    tools= tools
)


Testign the agent

In [ ]:

response = agent.invoke({
    "messages": [("human", "Calculate the power of Five and two")]
})

print(response["messages"][-1].content)